# Train a Simple Audio Recognition Model

This notebook demonstrates how to train a 20 kB [Simple Audio Recognition](https://www.tensorflow.org/tutorials/sequences/audio_recognition) model to recognize keywords in speech.

The model created in this notebook is used in the [micro_speech](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech) example for [TensorFlow Lite for MicroControllers](https://www.tensorflow.org/lite/microcontrollers/overview).

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech/train/train_micro_speech_model.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech/train/train_micro_speech_model.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>


**Training is much faster using GPU acceleration.** Before you proceed, ensure you are using a GPU runtime by going to **Runtime -> Change runtime type** and set **Hardware accelerator: GPU**. Training 15,000 iterations will take 1.5 - 2 hours on a GPU runtime.

## Configure Defaults

**MODIFY** the following constants for your specific use case.

In [ ]:
# A comma-delimited list of the words you want to train for.
# The options are: yes,no,up,down,left,right,on,off,stop,go
# All the other words will be used to train an "unknown" label and silent
# audio data with no spoken words will be used to train a "silence" label.
WANTED_WORDS = "right,left,six,eight"



# The number of steps and learning rates can be specified as comma-separated
# lists to define the rate at each stage. For example,
# TRAINING_STEPS=12000,3000 and LEARNING_RATE=0.001,0.0001
# will run 12,000 training loops in total, with a rate of 0.001 for the first
# 8,000, and 0.0001 for the final 3,000.
TRAINING_STEPS = "12000,3000"
LEARNING_RATE = "0.001,0.0001"

# Calculate the total number of steps, which is used to identify the checkpoint
# file name.
TOTAL_STEPS = str(sum(map(lambda string: int(string), TRAINING_STEPS.split(","))))

# Print the configuration to confirm it
print("Training these words: %s" % WANTED_WORDS)
print("Training steps in each stage: %s" % TRAINING_STEPS)
print("Learning rate in each stage: %s" % LEARNING_RATE)
print("Total number of training steps: %s" % TOTAL_STEPS)

**DO NOT MODIFY** the following constants as they include filepaths used in this notebook and data that is shared during training and inference.

In [ ]:
# Calculate the percentage of 'silence' and 'unknown' training samples required
# to ensure that we have equal number of samples for each label.
number_of_labels = WANTED_WORDS.count(',') + 1
number_of_total_labels = number_of_labels + 2 # for 'silence' and 'unknown' label
equal_percentage_of_training_samples = int(100.0/(number_of_total_labels))
SILENT_PERCENTAGE = equal_percentage_of_training_samples
UNKNOWN_PERCENTAGE = equal_percentage_of_training_samples

# Constants which are shared during training and inference
PREPROCESS = 'micro'
WINDOW_STRIDE = 20
MODEL_ARCHITECTURE = 'tiny_conv' # Other options include: single_fc, conv,
                      # low_latency_conv, low_latency_svdf, tiny_embedding_conv

# Constants used during training only
VERBOSITY = 'WARN'
EVAL_STEP_INTERVAL = '1000'
SAVE_STEP_INTERVAL = '1000'

# Constants for training directories and filepaths
DATASET_DIR =  'dataset/'
LOGS_DIR = 'logs/'
TRAIN_DIR = 'train/' # for training checkpoints and other files.

# Constants for inference directories and filepaths
import os
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
  os.mkdir(MODELS_DIR)
MODEL_TF = os.path.join(MODELS_DIR, 'model.pb')
MODEL_TFLITE = os.path.join(MODELS_DIR, 'model.tflite')
FLOAT_MODEL_TFLITE = os.path.join(MODELS_DIR, 'float_model.tflite')
MODEL_TFLITE_MICRO = os.path.join(MODELS_DIR, 'model.cc')
SAVED_MODEL = os.path.join(MODELS_DIR, 'saved_model')

QUANT_INPUT_MIN = 0.0
QUANT_INPUT_MAX = 26.0
QUANT_INPUT_RANGE = QUANT_INPUT_MAX - QUANT_INPUT_MIN

## Setup Environment

Install Dependencies

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


**DELETE** any old data from previous runs


In [ ]:
# !rm -rf {DATASET_DIR} {LOGS_DIR} {TRAIN_DIR} {MODELS_DIR}

import os
import shutil

# 要删除的目录列表
dirs_to_delete = [LOGS_DIR, MODELS_DIR]

for dir_path in dirs_to_delete:
    if os.path.exists(dir_path):
        try:
            shutil.rmtree(dir_path)
            print(f"✅ 已删除: {dir_path}")
        except Exception as e:
            print(f"❌ 删除失败 {dir_path}: {e}")
    else:
        print(f"⏭️ 目录不存在，跳过: {dir_path}")

print("\n清理完成！")

Clone the TensorFlow Github Repository, which contains the relevant code required to run this tutorial.

In [ ]:
# !git clone -q --depth 1 https://github.com/tensorflow/tensorflow

Load TensorBoard to visualize the accuracy and loss as training proceeds.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOGS_DIR}

## Training

The following script downloads the dataset and begin training.

In [ ]:
!python speech_commands/train.py \
--data_dir={DATASET_DIR} \
--wanted_words={WANTED_WORDS} \
--silence_percentage={SILENT_PERCENTAGE} \
--unknown_percentage={UNKNOWN_PERCENTAGE} \
--preprocess={PREPROCESS} \
--window_stride={WINDOW_STRIDE} \
--model_architecture={MODEL_ARCHITECTURE} \
--how_many_training_steps={TRAINING_STEPS} \
--learning_rate={LEARNING_RATE} \
--train_dir={TRAIN_DIR} \
--summaries_dir={LOGS_DIR} \
--verbosity={VERBOSITY} \
--eval_step_interval={EVAL_STEP_INTERVAL} \
--save_step_interval={SAVE_STEP_INTERVAL}

# Skipping the training

If you don't want to spend an hour or two training the model from scratch, you can download pretrained checkpoints by uncommenting the lines below (removing the '#'s at the start of each line) and running them.

In [ ]:
#!curl -O "https://storage.googleapis.com/download.tensorflow.org/models/tflite/speech_micro_train_2020_05_10.tgz"
#!tar xzf speech_micro_train_2020_05_10.tgz

## Generate a TensorFlow Model for Inference

Combine relevant training results (graph, weights, etc) into a single file for inference. This process is known as freezing a model and the resulting model is known as a frozen model/graph, as it cannot be further re-trained after this process.

In [ ]:
# !rm -rf {SAVED_MODEL}
# !python speech_commands/freeze.py \
# --wanted_words=$WANTED_WORDS \
# --window_stride_ms=$WINDOW_STRIDE \
# --preprocess=$PREPROCESS \
# --model_architecture=$MODEL_ARCHITECTURE \
# --start_checkpoint=$TRAIN_DIR$MODEL_ARCHITECTURE'.ckpt-'{TOTAL_STEPS} \
# --save_format=saved_model \
# --output_file={SAVED_MODEL}

import os
import shutil

# 1. 删除旧的 saved_model（兼容 Windows）
if os.path.exists(SAVED_MODEL):
    shutil.rmtree(SAVED_MODEL)
    print(f"✅ 已删除旧的 {SAVED_MODEL}")

# 2. 构造正确的 checkpoint 路径
checkpoint_path = os.path.join(TRAIN_DIR, f"{MODEL_ARCHITECTURE}.ckpt-{TOTAL_STEPS}")
print(f"Checkpoint 路径: {checkpoint_path}")

# 3. 检查 checkpoint 是否存在
if not os.path.exists(checkpoint_path + ".index"):
    print(f"❌ 错误: checkpoint 文件不存在!")
    print(f"请检查 {checkpoint_path}.index 是否存在")
    # 列出 train 目录内容帮助诊断
    if os.path.exists(TRAIN_DIR):
        print(f"\n{TRAIN_DIR} 目录内容:")
        for f in os.listdir(TRAIN_DIR):
            print(f"  {f}")
    else:
        print(f"❌ {TRAIN_DIR} 目录不存在!")
else:
    print(f"✅ Checkpoint 文件存在")

    # 4. 运行 freeze.py
    !python speech_commands/freeze.py \
    --wanted_words={WANTED_WORDS} \
    --window_stride_ms={WINDOW_STRIDE} \
    --preprocess={PREPROCESS} \
    --model_architecture={MODEL_ARCHITECTURE} \
    --start_checkpoint={checkpoint_path} \
    --save_format=saved_model \
    --output_file={SAVED_MODEL}
    
    # 5. 验证生成结果
    if os.path.exists(SAVED_MODEL):
        print(f"\n✅ 模型冻结成功！")
        print(f"SavedModel 位置: {SAVED_MODEL}")
        print("目录内容:")
        for f in os.listdir(SAVED_MODEL):
            print(f"  {f}")
    else:
        print(f"\n❌ 模型冻结失败，{SAVED_MODEL} 未生成")

## Generate a TensorFlow Lite Model

Convert the frozen graph into a TensorFlow Lite model, which is fully quantized for use with embedded devices.

The following cell will also print the model size, which will be under 20 kilobytes.

In [ ]:
import sys
# We add this path so we can import the speech processing modules.
sys.path.append("speech_commands/")
import input_data
import models
import numpy as np

In [ ]:
SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
WINDOW_SIZE_MS = 30.0
FEATURE_BIN_COUNT = 40
BACKGROUND_FREQUENCY = 0.8
BACKGROUND_VOLUME_RANGE = 0.1
TIME_SHIFT_MS = 100.0

DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
VALIDATION_PERCENTAGE = 10
TESTING_PERCENTAGE = 10

In [ ]:
model_settings = models.prepare_model_settings(
    len(input_data.prepare_words_list(WANTED_WORDS.split(','))),
    SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
    WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)
audio_processor = input_data.AudioProcessor(
    DATA_URL, DATASET_DIR,
    SILENT_PERCENTAGE, UNKNOWN_PERCENTAGE,
    WANTED_WORDS.split(','), VALIDATION_PERCENTAGE,
    TESTING_PERCENTAGE, model_settings, LOGS_DIR)

In [ ]:
with tf.compat.v1.Session() as sess:
  float_converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
  float_tflite_model = float_converter.convert()
  float_tflite_model_size = open(FLOAT_MODEL_TFLITE, "wb").write(float_tflite_model)
  print("Float model is %d bytes" % float_tflite_model_size)

  converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
  converter.optimizations = [tf.lite.Optimize.DEFAULT]
  converter.inference_input_type = tf.int8
  converter.inference_output_type = tf.int8
  def representative_dataset_gen():
    for i in range(100):
      data, _ = audio_processor.get_data(1, i*1, model_settings,
                                         BACKGROUND_FREQUENCY,
                                         BACKGROUND_VOLUME_RANGE,
                                         TIME_SHIFT_MS,
                                         'testing',
                                         sess)
      flattened_data = np.array(data.flatten(), dtype=np.float32).reshape(1, 1960)
      yield [flattened_data]
  converter.representative_dataset = representative_dataset_gen
  tflite_model = converter.convert()
  tflite_model_size = open(MODEL_TFLITE, "wb").write(tflite_model)
  print("Quantized model is %d bytes" % tflite_model_size)


## Testing the TensorFlow Lite model's accuracy

Verify that the model we've exported is still accurate, using the TF Lite Python API and our test set.

In [ ]:
# Helper function to run inference
def run_tflite_inference(tflite_model_path, model_type="Float"):
  # Load test data
  np.random.seed(0) # set random seed for reproducible test results.
  with tf.compat.v1.Session() as sess:
    test_data, test_labels = audio_processor.get_data(
        -1, 0, model_settings, BACKGROUND_FREQUENCY, BACKGROUND_VOLUME_RANGE,
        TIME_SHIFT_MS, 'testing', sess)
  test_data = np.expand_dims(test_data, axis=1).astype(np.float32)

  # Initialize the interpreter
  interpreter = tf.lite.Interpreter(tflite_model_path,
                                    experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_REF)
  interpreter.allocate_tensors()

  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]

  # For quantized models, manually quantize the input data from float to integer
  if model_type == "Quantized":
    input_scale, input_zero_point = input_details["quantization"]
    test_data = test_data / input_scale + input_zero_point
    test_data = test_data.astype(input_details["dtype"])

  correct_predictions = 0
  for i in range(len(test_data)):
    interpreter.set_tensor(input_details["index"], test_data[i])
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])[0]
    top_prediction = output.argmax()
    correct_predictions += (top_prediction == test_labels[i])

  print('%s model accuracy is %f%% (Number of test samples=%d)' % (
      model_type, (correct_predictions * 100) / len(test_data), len(test_data)))

In [ ]:
# Compute float model accuracy
run_tflite_inference(FLOAT_MODEL_TFLITE)

# Compute quantized model accuracy
run_tflite_inference(MODEL_TFLITE, model_type='Quantized')

## Generate a TensorFlow Lite for MicroControllers Model
Convert the TensorFlow Lite model into a C source file that can be loaded by TensorFlow Lite for Microcontrollers.

In [ ]:
# # Install xxd if it is not available
# !apt-get update && apt-get -qq install xxd
# # Convert to a C source file
# !xxd -i {MODEL_TFLITE} > {MODEL_TFLITE_MICRO}
# # Update variable names
# REPLACE_TEXT = MODEL_TFLITE.replace('/', '_').replace('.', '_')
# !sed -i 's/'{REPLACE_TEXT}'/g_model/g' {MODEL_TFLITE_MICRO}

import os
import re

MODEL_TFLITE = "models/model.tflite"  # 根据你的实际路径调整
MODEL_TFLITE_MICRO = "models/model.cc"  # 根据你的实际路径调整

# 1. 检查模型文件是否存在
if not os.path.exists(MODEL_TFLITE):
    print(f"❌ 错误: {MODEL_TFLITE} 不存在！")
    print("请先运行前面的步骤生成 TFLite 模型")
else:
    print(f"✅ 找到模型文件: {MODEL_TFLITE} ({os.path.getsize(MODEL_TFLITE)} 字节)")
    
    # 2. 读取二进制模型文件并转换为 C 数组
    with open(MODEL_TFLITE, "rb") as f:
        model_data = f.read()
    
    # 生成 C 数组
    c_array_name = "g_model"
    c_lines = [
        f"// Auto-generated from {MODEL_TFLITE}\n",
        f"// Model size: {len(model_data)} bytes\n",
        f"#include <stdint.h>\n",
        f"#include <stddef.h>\n",
        f"\n",
        f"const unsigned char {c_array_name}[] = {{\n"
    ]
    
    # 每行 12 个字节，格式化输出
    for i in range(0, len(model_data), 12):
        chunk = model_data[i:i+12]
        hex_str = ", ".join([f"0x{b:02x}" for b in chunk])
        c_lines.append(f"  {hex_str},\n")
    
    c_lines.append("};\n")
    c_lines.append(f"const size_t {c_array_name}_len = sizeof({c_array_name});\n")
    
    # 3. 写入 C 文件
    with open(MODEL_TFLITE_MICRO, "w") as f:
        f.writelines(c_lines)
    
    print(f"✅ 已生成 C 数组文件: {MODEL_TFLITE_MICRO}")
    print(f"   数组名: {c_array_name}")
    print(f"   数组长度: {len(model_data)} 字节")
    
    # 4. 显示文件头部预览
    print("\n--- 文件内容预览（前 20 行）---")
    with open(MODEL_TFLITE_MICRO, "r") as f:
        lines = f.readlines()
        for line in lines[:20]:
            print(line.rstrip())
        if len(lines) > 20:
            print(f"... (共 {len(lines)} 行)")

## Deploy to a Microcontroller

Follow the instructions in the [micro_speech](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech) README.md for [TensorFlow Lite for MicroControllers](https://www.tensorflow.org/lite/microcontrollers/overview) to deploy this model on a specific microcontroller.

**Reference Model:** If you have not modified this notebook, you can follow the instructions as is, to deploy the model. Refer to the [`micro_speech/train/models`](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech/train/models) directory to access the models generated in this notebook.

**New Model:** If you have generated a new model to identify different words: (i) Update `kCategoryCount` and `kCategoryLabels` in [`micro_speech/micro_features/micro_model_settings.h`](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech/micro_features/micro_model_settings.h) and (ii) Update the values assigned to the variables defined in [`micro_speech/micro_features/model.cc`](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech/micro_features/model.cc) with values displayed after running the following cell.

In [ ]:
# Print the C source file
# !cat {MODEL_TFLITE_MICRO}
import os

MODEL_TFLITE_MICRO = "models/model.cc"  # 根据你的实际路径调整

if os.path.exists(MODEL_TFLITE_MICRO):
    print(f"=== 文件内容: {MODEL_TFLITE_MICRO} ===\n")
    
    # 查看整个文件
    with open(MODEL_TFLITE_MICRO, "r") as f:
        content = f.read()
        print(content)
    
    # 或者只查看前 100 行
    # with open(MODEL_TFLITE_MICRO, "r") as f:
    #     lines = f.readlines()
    #     for i, line in enumerate(lines[:100]):  # 只显示前 100 行
    #         print(f"{i+1}: {line.rstrip()}")
    #     if len(lines) > 100:
    #         print(f"... (共 {len(lines)} 行)")
else:
    print(f"❌ 文件不存在: {MODEL_TFLITE_MICRO}")